In [1]:
import numpy as np  # NumPy for numerical operations
import pandas as pd  # Pandas for da.detectta manipulation
import os
from google.colab import files

In [2]:
import os
print('Current directory contents:', os.listdir('.'))
if os.path.exists('/content'):
    print('/content contents:', os.listdir('/content'))

# Look for any directory that might contain the CSVs
for root, dirs, files in os.walk('/content'):
    if any(f.endswith('.csv') for f in files):
        print(f'Found CSVs in: {root} (Count: {len([f for f in files if f.endswith(".csv")])})')

Current directory contents: ['.config', 'sample_data']
/content contents: ['.config', 'sample_data']
Found CSVs in: /content/sample_data (Count: 4)


In [3]:
import os
print('Current directory contents:', os.listdir('.'))
# Look for any directory that might contain the CSVs
for root, dirs, files in os.walk('/content'):
    csv_files = [f for f in files if f.endswith('.csv')]
    if csv_files:
        print(f'Found {len(csv_files)} CSVs in: {root}')

Current directory contents: ['.config', 'sample_data']
Found 4 CSVs in: /content/sample_data


In [4]:
from google.colab import files

# file "https://www.kaggle.com/settings"
files.upload() #upload kaggle.json (Legacy API Credentials)

!mkdir -p ~/.kaggle/

# Move the uploaded kaggle.json file to the .kaggle directory
!mv kaggle.json ~/.kaggle/

# Set secure permissions for the API key file (read-only for owner)
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c super-ai-engineer-ss-6-sleep-stage-classification

Saving kaggle.json to kaggle.json
100% 2.13G/2.13G [02:28<00:00, 15.4MB/s]



In [5]:
! unzip super-ai-engineer-ss-6-sleep-stage-classification.zip

Streaming output truncated to the last 5000 lines.
  inflating: test_segment/test_segment/test004/test004_00448.csv  
  inflating: test_segment/test_segment/test004/test004_00449.csv  
  inflating: test_segment/test_segment/test004/test004_00450.csv  
  inflating: test_segment/test_segment/test004/test004_00451.csv  
  inflating: test_segment/test_segment/test004/test004_00452.csv  
  inflating: test_segment/test_segment/test004/test004_00453.csv  
  inflating: test_segment/test_segment/test004/test004_00454.csv  
  inflating: test_segment/test_segment/test004/test004_00455.csv  
  inflating: test_segment/test_segment/test004/test004_00456.csv  
  inflating: test_segment/test_segment/test004/test004_00457.csv  
  inflating: test_segment/test_segment/test004/test004_00458.csv  
  inflating: test_segment/test_segment/test004/test004_00459.csv  
  inflating: test_segment/test_segment/test004/test004_00460.csv  
  inflating: test_segment/test_segment/test004/test004_00461.csv  
  inflating

In [6]:
import os
import numpy as np
import pandas as pd

def read_all_test_files(base_path):
    all_data = []
    if not os.path.exists(base_path):
        print(f"Error: Path {base_path} not found.")
        return pd.DataFrame()
    files = sorted([f for f in os.listdir(base_path) if f.endswith('.csv')])
    for file_name in files:
        file_path = os.path.join(base_path, file_name)
        df = pd.read_csv(file_path)
        patient_id = os.path.splitext(file_name)[0]
        df['patient_id'] = patient_id
        df['time'] = np.arange(len(df)) / 16.0
        all_data.append(df)
    if not all_data:
        return pd.DataFrame()
    return pd.concat(all_data, ignore_index=True)

print('Loading test data...')
test_path = '/content/test/test'
test_df = read_all_test_files(test_path)
print(f'Test data loaded. Shape: {test_df.shape}')

Loading test data...
Error: Path /content/test/test not found.
Test data loaded. Shape: (0, 0)


In [7]:
WINDOW_SIZE = 480

def create_test_segments(df):
    if df.empty:
        print("Input DataFrame is empty.")
        return pd.DataFrame()
    # Ensure the same feature engineering as training
    df['segment_id'] = df.groupby('patient_id').cumcount() // WINDOW_SIZE

    grouped = df.groupby(['patient_id', 'segment_id'])

    # Aggregate features: Mean and Std for sensors
    agg_df = grouped.agg({
        'BVP': ['mean', 'std'],
        'ACC_X': ['mean', 'std'],
        'ACC_Y': ['mean', 'std'],
        'ACC_Z': ['mean', 'std'],
        'TEMP': ['mean', 'std'],
        'EDA': ['mean', 'std'],
        'HR': ['mean', 'std'],
        'IBI': ['mean', 'std']
    })

    # Flatten multi-index
    agg_df.columns = [f'{col[0]}_{col[1]}' for col in agg_df.columns]
    return agg_df.reset_index()

print('Processing test segments...')
test_segments = create_test_segments(test_df)
if not test_segments.empty:
    display(test_segments.head())

Processing test segments...
Input DataFrame is empty.


In [8]:
def read_all_train_files_to_dataframe(base_path):
    all_data = []
    for file_name in sorted(os.listdir(base_path)):
        if file_name.endswith('.csv'):
            file_path = os.path.join(base_path, file_name)
            df = pd.read_csv(file_path)
            # Extract patient_id from filename (e.g., 'train058.csv' -> 'train058')
            patient_id = os.path.splitext(file_name)[0]
            df['patient_id'] = patient_id
            all_data.append(df)
    return pd.concat(all_data, ignore_index=True)

# The current working directory is '/content/train/train'
train_df = read_all_train_files_to_dataframe("/content/train/train")

In [9]:
# Move 'patient_id' to the first column, and 'time' next to it
current_cols = train_df.columns.tolist()

# Add a 'time' column: 1 second = 16 data points
train_df['time'] = train_df.groupby('patient_id').cumcount() / 16.0

# Ensure 'patient_id' and 'time' are at the beginning
current_cols_after_time_addition = train_df.columns.tolist()
current_cols_after_time_addition.remove('patient_id')
current_cols_after_time_addition.remove('time')
new_col_order = ['patient_id', 'time'] + current_cols_after_time_addition
train_df = train_df[new_col_order]


# Display the first few rows with the new column order and 'time' column
display(train_df.head())

,patient_id,time,BVP,ACC_X,ACC_Y,ACC_Z,TEMP,EDA,HR,IBI,Sleep_Stage
0,train001,0.0000,25.325870,-21.809247,-60.302750,4.940839,31.722653,0.064595,72.015570,1.050338,W
1,train001,0.0625,20.021505,-19.437787,-60.565345,7.408788,31.722647,0.064523,72.015802,1.050338,W
2,train001,0.1250,16.314478,-21.624667,-61.142561,4.105717,31.722735,0.064659,72.017417,1.050338,W
3,train001,0.1875,9.324392,-21.761314,-61.985822,3.972967,31.722564,0.064440,72.013801,1.050338,W
4,train001,0.2500,-1.014338,-19.055301,-59.934137,9.097628,31.722790,0.065397,72.018920,1.050338,W


## RandomForestClassifier

---



In [10]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

WINDOW_SIZE = 480
# Detect paths dynamically based on common unzipping patterns
TRAIN_PATH = '/content/train/train' if os.path.exists('/content/train/train') else '/content/train'
TEST_PATH = '/content/test/test' if os.path.exists('/content/test/test') else '/content/test'

def load_and_segment(base_path, is_train=True):
    if not os.path.exists(base_path):
        print(f'Warning: {base_path} not found.')
        return pd.DataFrame()

    all_segments = []
    files = sorted([f for f in os.listdir(base_path) if f.endswith('.csv')])

    print(f'Processing {len(files)} files from {base_path}...')

    for file_name in files:
        df = pd.read_csv(os.path.join(base_path, file_name))
        patient_id = os.path.splitext(file_name)[0]
        df['segment_id'] = np.arange(len(df)) // WINDOW_SIZE

        agg_rules = {
            'BVP': ['mean', 'std', 'min', 'max', 'median'],
            'ACC_X': ['mean', 'std', 'min', 'max', 'median'],
            'ACC_Y': ['mean', 'std', 'min', 'max', 'median'],
            'ACC_Z': ['mean', 'std', 'min', 'max', 'median'],
            'TEMP': ['mean', 'std', 'min', 'max', 'median'],
            'EDA': ['mean', 'std', 'min', 'max', 'median'],
            'HR': ['mean', 'std', 'min', 'max', 'median'],
            'IBI': ['mean', 'std', 'min', 'max', 'median']
        }

        if is_train and 'Sleep_Stage' in df.columns:
            agg_rules['Sleep_Stage'] = 'first'

        seg_df = df.groupby('segment_id').agg(agg_rules)
        seg_df.columns = [f'{c[0]}_{c[1]}' if isinstance(c, tuple) else c for c in seg_df.columns]
        seg_df['patient_id'] = patient_id
        all_segments.append(seg_df)

    if not all_segments: return pd.DataFrame()
    return pd.concat(all_segments, ignore_index=True)

In [11]:
# 1. Load and Process Training Data
train_seg = load_and_segment(TRAIN_PATH, is_train=True)

# 2. Encode Labels
le = LabelEncoder()
y = le.fit_transform(train_seg['Sleep_Stage_first'])
X = train_seg.drop(['Sleep_Stage_first', 'patient_id'], axis=1)

# 3. Train/Val Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 4. Train Model
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42, class_weight='balanced') # Added class_weight
rf.fit(X_train, y_train)

# 5. Evaluate
y_pred = rf.predict(X_val)
print(f"Weighted F1 Score: {f1_score(y_val, y_pred, average='weighted'):.4f}")
print(classification_report(y_val, y_pred, target_names=le.classes_))

Processing 83 files from /content/train/train...
Weighted F1 Score: 0.7841
              precision    recall  f1-score   support

          N1       0.58      0.28      0.38      1550
          N2       0.81      0.93      0.86      6757
          N3       0.81      0.70      0.75       469
           R       0.90      0.85      0.87      1407
           W       0.79      0.77      0.78      3166

    accuracy                           0.80     13349
   macro avg       0.78      0.71      0.73     13349
weighted avg       0.79      0.80      0.78     13349



In [21]:
# 1. Load and Process Test Data from nested subdirectories
import os
import pandas as pd
import numpy as np

def load_nested_test_data(root_path):
    all_segments = []
    # Walk through the test_segment directory to find all CSV files
    for root, dirs, files in os.walk(root_path):
        csv_files = [f for f in files if f.endswith('.csv')]
        if not csv_files:
            continue

        # Sort to maintain some internal order if needed, though sample_sub governs final index
        for file_name in sorted(csv_files):
            df = pd.read_csv(os.path.join(root, file_name))
            patient_id = os.path.splitext(file_name)[0]
            df['segment_id'] = np.arange(len(df)) // WINDOW_SIZE

            agg_rules = {
                'BVP': ['mean', 'std', 'min', 'max', 'median'],
                'ACC_X': ['mean', 'std', 'min', 'max', 'median'],
                'ACC_Y': ['mean', 'std', 'min', 'max', 'median'],
                'ACC_Z': ['mean', 'std', 'min', 'max', 'median'],
                'TEMP': ['mean', 'std', 'min', 'max', 'median'],
                'EDA': ['mean', 'std', 'min', 'max', 'median'],
                'HR': ['mean', 'std', 'min', 'max', 'median'],
                'IBI': ['mean', 'std', 'min', 'max', 'median']
            }

            seg_df = df.groupby('segment_id').agg(agg_rules)
            # Flatten column names to match training: Sensor_Stat
            seg_df.columns = [f'{c[0]}_{c[1]}' for c in seg_df.columns]
            seg_df['patient_id'] = patient_id
            all_segments.append(seg_df)

    if not all_segments: return pd.DataFrame()
    return pd.concat(all_segments, ignore_index=True)

print('Processing nested test files...')
test_seg = load_nested_test_data('/content/test_segment')

if not test_seg.empty:
    # 2. Predict using the RandomForest model 'rf'
    X_test = test_seg.drop(['patient_id'], axis=1)
    test_preds = rf.predict(X_test)
    test_labels = le.inverse_transform(test_preds)

    # 3. Create Submission
    sample_sub = pd.read_csv('sample_submission.csv')

    if len(test_labels) == len(sample_sub):
        submission = pd.DataFrame({
            'id': sample_sub['id'], # Corrected from 'index' to 'id'
            'Sleep_Stage': test_labels
        })
        submission.to_csv('submission.csv', index=False)
        print('Submission saved successfully!')
        display(submission.head())
    else:
        print(f'Warning: Prediction length ({len(test_labels)}) != Sample Submission length ({len(sample_sub)})')
else:
    print('Test data could not be loaded from /content/test_segment.')

Processing nested test files...
Submission saved successfully!


,id,Sleep_Stage
0,test001_00000,W
1,test001_00001,W
2,test001_00002,W
3,test001_00003,W
4,test001_00004,W


Public Score 0.38325

##  XGBoost or LightGBM to

In [20]:
import xgboost as xgb
import lightgbm as lgb

print('Training XGBoost Classifier...')
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax', # For multi-class classification
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    n_estimators=100,
    n_jobs=-1,
    random_state=42,
    tree_method='hist' # For faster training
)
xgb_model.fit(X_train, y_train)

print('Evaluating XGBoost Classifier...')
y_pred_xgb = xgb_model.predict(X_val)
print(f"XGBoost Weighted F1 Score: {f1_score(y_val, y_pred_xgb, average='weighted'):.4f}")
print(classification_report(y_val, y_pred_xgb, target_names=le.classes_))

print('\nTraining LightGBM Classifier...')
lgbm_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(le.classes_),
    n_estimators=100,
    n_jobs=-1,
    random_state=42
)
lgbm_model.fit(X_train, y_train)

print('Evaluating LightGBM Classifier...')
y_pred_lgbm = lgbm_model.predict(X_val)
print(f"LightGBM Weighted F1 Score: {f1_score(y_val, y_pred_lgbm, average='weighted'):.4f}")
print(classification_report(y_val, y_pred_lgbm, target_names=le.classes_))

Training XGBoost Classifier...
Evaluating XGBoost Classifier...
XGBoost Weighted F1 Score: 0.7349
              precision    recall  f1-score   support

          N1       0.51      0.21      0.29      1550
          N2       0.76      0.91      0.83      6757
          N3       0.78      0.67      0.72       469
           R       0.87      0.71      0.78      1407
           W       0.76      0.72      0.74      3166

    accuracy                           0.75     13349
   macro avg       0.73      0.64      0.67     13349
weighted avg       0.74      0.75      0.73     13349


Training LightGBM Classifier...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.053631 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10200
[LightGBM] [Info] Number of data points in the train set: 53396, number of used features: 40
[LightGBM] [Info] Start training from score -2.152703
[LightGBM] [Info] Start training fro

In [16]:
# 1. Load and Process Test Data from nested subdirectories

print('Processing nested test files...')
test_seg = load_nested_test_data('/content/test_segment')

if not test_seg.empty:
    # 2. Predict using the LightGBM model 'lgbm_model'
    X_test = test_seg.drop(['patient_id'], axis=1)
    test_preds = xgb_model.predict(X_test)
    test_labels = le.inverse_transform(test_preds)

    # 3. Create Submission
    sample_sub = pd.read_csv('sample_submission.csv')

    if len(test_labels) == len(sample_sub):
        submission = pd.DataFrame({
            'id': sample_sub['id'],
            'Sleep_Stage': test_labels
        })
        submission.to_csv('submission.csv', index=False)
        print('Submission saved successfully!')
        display(submission.head())
    else:
        print(f'Warning: Prediction length ({len(test_labels)}) != Sample Submission length ({len(sample_sub)})')
else:
    print('Test data could not be loaded from /content/test_segment.')

Processing nested test files...
Submission saved successfully!


,id,Sleep_Stage
0,test001_00000,W
1,test001_00001,W
2,test001_00002,N2
3,test001_00003,W
4,test001_00004,W


Public Score 0.35474

### XGBoost Hyperparameter Tuning with RandomizedSearchCV

### LightGBM Hyperparameter Tuning with RandomizedSearchCV

To further improve the performance, especially for the N1 class, we will tune the LightGBM model's hyperparameters using `RandomizedSearchCV`.

We will define a parameter distribution to explore key hyperparameters like `num_leaves`, `max_depth`, `learning_rate`, `n_estimators`, `subsample`, and `colsample_bytree`. We will optimize for the weighted F1-score during the search.

In [15]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'num_leaves': randint(20, 60),  # Number of leaves in one tree
    'max_depth': randint(5, 15),     # Maximum tree depth
    'learning_rate': uniform(0.01, 0.1), # Step size shrinkage to prevent overfitting
    'n_estimators': randint(100, 500), # Number of boosting rounds
    'subsample': uniform(0.6, 0.4),  # Subsample ratio of the training instance
    'colsample_bytree': uniform(0.6, 0.4), # Subsample ratio of columns when constructing each tree
    'reg_alpha': uniform(0, 0.5),    # L1 regularization term
    'reg_lambda': uniform(0, 0.5),   # L2 regularization term
    'min_child_samples': randint(20, 100), # Minimum number of data needed in a child (leaf)
}

# Initialize LightGBM model with class_weight 'balanced'
lgbm = lgb.LGBMClassifier(objective='multiclass', num_class=len(le.classes_), random_state=42, n_jobs=-1, class_weight='balanced')

# Initialize RandomizedSearchCV
# We use 'f1_weighted' as the scoring metric to account for class imbalance
# n_iter controls the number of random combinations to try
random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=10, # Reduced number of parameter settings that are sampled
    scoring='f1_weighted',
    cv=2, # Reduced 2-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1 # Use all available cores
)

print('Starting RandomizedSearchCV for LightGBM...')
random_search.fit(X_train, y_train)

print('\nBest parameters found by RandomizedSearchCV:')
print(random_search.best_params_)

# Get the best model
best_lgbm_model = random_search.best_estimator_

print('\nEvaluating the best LightGBM model on the validation set...')
y_pred_best_lgbm = best_lgbm_model.predict(X_val)
print(f"Best LightGBM Weighted F1 Score: {f1_score(y_val, y_pred_best_lgbm, average='weighted'):.4f}")
print(classification_report(y_val, y_pred_best_lgbm, target_names=le.classes_))

Starting RandomizedSearchCV for LightGBM...
Fitting 2 folds for each of 10 candidates, totalling 20 fits


KeyboardInterrupt: 

In [ ]:
print('Processing nested test files...')
test_seg = load_nested_test_data('/content/test_segment')

if not test_seg.empty:
    # 2. Predict using the LightGBM model 'lgbm_model'
    X_test = test_seg.drop(['patient_id'], axis=1)
    test_preds = best_lgbm_model.predict(X_test)
    test_labels = le.inverse_transform(test_preds)

    # 3. Create Submission
    sample_sub = pd.read_csv('sample_submission.csv')

    if len(test_labels) == len(sample_sub):
        submission = pd.DataFrame({
            'id': sample_sub['id'],
            'Sleep_Stage': test_labels
        })
        submission.to_csv('submission.csv', index=False)
        print('Submission saved successfully!')
        display(submission.head())
    else:
        print(f'Warning: Prediction length ({len(test_labels)}) != Sample Submission length ({len(sample_sub)})')
else:
    print('Test data could not be loaded from /content/test_segment.')

### Classification Report for Tuned RandomForestClassifier

In [ ]:
from sklearn.metrics import classification_report

# Ensure best_rf_model is available (from previous execution of cell e884b6c0)
if 'best_rf_model' in locals():
    y_pred_best_rf = best_rf_model.predict(X_val)
    print(f"Best RandomForestClassifier Weighted F1 Score: {f1_score(y_val, y_pred_best_rf, average='weighted'):.4f}")
    print(classification_report(y_val, y_pred_best_rf, target_names=le.classes_))
else:
    print("Tuned RandomForest model not found. Please run its tuning cell (e884b6c0) first.")

### Classification Report for Tuned LightGBM Classifier

In [ ]:
from sklearn.metrics import classification_report

# Ensure best_lgbm_model is available (from previous execution of cell 4dbb29d5)
if 'best_lgbm_model' in locals():
    y_pred_best_lgbm = best_lgbm_model.predict(X_val)
    print(f"Best LightGBM Weighted F1 Score: {f1_score(y_val, y_pred_best_lgbm, average='weighted'):.4f}")
    print(classification_report(y_val, y_pred_best_lgbm, target_names=le.classes_))
else:
    print("Tuned LightGBM model not found. Please run its tuning cell (4dbb29d5) first.")

## a 1D Convolutional Neural Network (1D-CNN)

In [25]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler

# Standardize features for the Neural Network
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Reshape data for 1D CNN: (samples, features, 1)
X_train_cnn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_val_cnn = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))

In [26]:
def build_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),

        layers.Conv1D(128, kernel_size=3, activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn((X_train_cnn.shape[1], 1), len(le.classes_))
cnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 38, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 38, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 19, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 19, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 17, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 17, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,309 (134.02 KB)

 Trainable params: 33,925 (132.52 KB)

 Non-trainable params: 384 (1.50 KB)

In [27]:
history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val_cnn, y_val),
    verbose=1
)

# Evaluate
y_pred_cnn = cnn_model.predict(X_val_cnn).argmax(axis=1)
print(f"CNN Weighted F1 Score: {f1_score(y_val, y_pred_cnn, average='weighted'):.4f}")
print(classification_report(y_val, y_pred_cnn, target_names=le.classes_))

Epoch 1/20
835/835 ━━━━━━━━━━━━━━━━━━━━ 49s 55ms/step - accuracy: 0.5553 - loss: 1.2184 - val_accuracy: 0.5668 - val_loss: 1.1575
Epoch 2/20
835/835 ━━━━━━━━━━━━━━━━━━━━ 29s 34ms/step - accuracy: 0.5703 - loss: 1.1554 - val_accuracy: 0.5752 - val_loss: 1.1324
Epoch 3/20
835/835 ━━━━━━━━━━━━━━━━━━━━ 26s 32ms/step - accuracy: 0.5755 - loss: 1.1294 - val_accuracy: 0.5857 - val_loss: 1.0933
Epoch 4/20
835/835 ━━━━━━━━━━━━━━━━━━━━ 25s 30ms/step - accuracy: 0.5777 - loss: 1.1095 - val_accuracy: 0.5848 - val_loss: 1.0675
Epoch 5/20
835/835 ━━━━━━━━━━━━━━━━━━━━ 24s 28ms/step - accuracy: 0.5850 - loss: 1.0920 - val_accuracy: 0.5956 - val_loss: 1.0525
Epoch 6/20
835/835 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.5878 - loss: 1.0764 - val_accuracy: 0.6024 - val_loss: 1.0344
Epoch 7/20
835/835 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.5910 - loss: 1.0602 - val_accuracy: 0.6008 - val_loss: 1.0415
Epoch 8/20
835/835 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.5956 - loss: 1.0507 - 

In [29]:
print('Processing nested test files...')
test_seg = load_nested_test_data('/content/test_segment')

if not test_seg.empty:
    # 1. Prepare features
    X_test = test_seg.drop(['patient_id'], axis=1).values

    # 2. Reshape for CNN: (samples, features, 1)
    # The error occurred because the model expected 3D input but got 2D
    X_test_cnn = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # 3. Predict using the CNN model
    test_preds_probs = cnn_model.predict(X_test_cnn)
    test_preds = test_preds_probs.argmax(axis=1)
    test_labels = le.inverse_transform(test_preds)

    # 4. Create Submission
    sample_sub = pd.read_csv('sample_submission.csv')

    if len(test_labels) == len(sample_sub):
        submission = pd.DataFrame({
            'id': sample_sub['id'],
            'Sleep_Stage': test_labels
        })
        submission.to_csv('submission_cnn.csv', index=False)
        print('CNN Submission saved successfully!')
        display(submission.head())
    else:
        print(f'Warning: Prediction length ({len(test_labels)}) != Sample Submission length ({len(sample_sub)})')
else:
    print('Test data could not be loaded from /content/test_segment.')

Processing nested test files...
245/245 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
CNN Submission saved successfully!


,id,Sleep_Stage
0,test001_00000,N2
1,test001_00001,N2
2,test001_00002,N2
3,test001_00003,N2
4,test001_00004,N2


## autoLM

In [10]:
!pip install flaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.7/337.7 kB 19.5 MB/s eta 0:00:00


In [22]:
from flaml import AutoML
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import os
import numpy as np
import pandas as pd

# --- Required Definitions to make cell self-contained ---
WINDOW_SIZE = 480
TRAIN_PATH = '/content/train/train' if os.path.exists('/content/train/train') else '/content/train'

def load_and_segment(base_path, is_train=True):
    if not os.path.exists(base_path):
        print(f'Warning: {base_path} not found.')
        return pd.DataFrame()
    all_segments = []
    files = sorted([f for f in os.listdir(base_path) if f.endswith('.csv')])
    for file_name in files:
        df = pd.read_csv(os.path.join(base_path, file_name))
        patient_id = os.path.splitext(file_name)[0]
        df['segment_id'] = np.arange(len(df)) // WINDOW_SIZE
        agg_rules = {
            'BVP': ['mean', 'std', 'min', 'max', 'median'],
            'ACC_X': ['mean', 'std', 'min', 'max', 'median'],
            'ACC_Y': ['mean', 'std', 'min', 'max', 'median'],
            'ACC_Z': ['mean', 'std', 'min', 'max', 'median'],
            'TEMP': ['mean', 'std', 'min', 'max', 'median'],
            'EDA': ['mean', 'std', 'min', 'max', 'median'],
            'HR': ['mean', 'std', 'min', 'max', 'median'],
            'IBI': ['mean', 'std', 'min', 'max', 'median']
        }
        if is_train and 'Sleep_Stage' in df.columns:
            agg_rules['Sleep_Stage'] = 'first'
        seg_df = df.groupby('segment_id').agg(agg_rules)
        seg_df.columns = [f'{c[0]}_{c[1]}' if isinstance(c, tuple) else c for c in seg_df.columns]
        seg_df['patient_id'] = patient_id
        all_segments.append(seg_df)
    return pd.concat(all_segments, ignore_index=True) if all_segments else pd.DataFrame()

# 1. Load and Process Data
print('Loading data...')
train_seg = load_and_segment(TRAIN_PATH, is_train=True)
le = LabelEncoder()
y_encoded = le.fit_transform(train_seg['Sleep_Stage_first'])
X = train_seg.drop(['Sleep_Stage_first', 'patient_id'], axis=1)

# Use raw strings for FLAML classification labels to avoid binary metric errors
y = train_seg['Sleep_Stage_first'].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 2. Initialize AutoML
automl = AutoML()

# 3. Define settings - explicitly set metric to 'macro_f1' or similar for multiclass
settings = {
    "time_budget": 300,  # 5 minutes for testing
    "metric": 'macro_f1',
    "task": 'classification',
    "log_file_name": 'flaml.log',
    "seed": 42,
}

print('Starting FLAML search...')
# 4. Train with FLAML
automl.fit(X_train=X_train, y_train=y_train, **settings)

# 5. Predict and evaluate
y_pred_flaml = automl.predict(X_val)
print(f'\nBest ML learner: {automl.best_estimator}')
print(f'FLAML Val Weighted F1: {f1_score(y_val, y_pred_flaml, average="weighted"):.4f}')
print(classification_report(y_val, y_pred_flaml))

Loading data...
Starting FLAML search...
[flaml.automl.logger: 04-04 10:34:54] {2375} INFO - task = classification
[flaml.automl.logger: 04-04 10:34:54] {2386} INFO - Evaluation method: holdout
[flaml.automl.logger: 04-04 10:34:54] {2489} INFO - Minimizing error metric: 1-macro_f1
[flaml.automl.logger: 04-04 10:34:54] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 04-04 10:34:54] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 04-04 10:34:54] {3046} INFO - Estimated sufficient time budget=6713s. Estimated necessary time budget=155s.
[flaml.automl.logger: 04-04 10:34:54] {3097} INFO -  at 0.7s,	estimator lgbm's best error=7.8952e-01,	best estimator lgbm's best error=7.8952e-01
[flaml.automl.logger: 04-04 10:34:54] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 04-04 10:34:54] {3097} INFO -  at 0.9s,	estimator lgbm's best error=7.8952e-01,	best est

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 04-04 10:34:55] {3097} INFO -  at 1.7s,	estimator sgd's best error=8.6557e-01,	best estimator lgbm's best error=7.8952e-01
[flaml.automl.logger: 04-04 10:34:55] {2911} INFO - iteration 3, current learner lgbm
[flaml.automl.logger: 04-04 10:34:56] {3097} INFO -  at 2.5s,	estimator lgbm's best error=7.4096e-01,	best estimator lgbm's best error=7.4096e-01
[flaml.automl.logger: 04-04 10:34:56] {2911} INFO - iteration 4, current learner xgboost
[flaml.automl.logger: 04-04 10:34:57] {3097} INFO -  at 3.3s,	estimator xgboost's best error=8.6557e-01,	best estimator lgbm's best error=7.4096e-01
[flaml.automl.logger: 04-04 10:34:57] {2911} INFO - iteration 5, current learner extra_tree
[flaml.automl.logger: 04-04 10:34:57] {3097} INFO -  at 3.4s,	estimator extra_tree's best error=8.0100e-01,	best estimator lgbm's best error=7.4096e-01
[flaml.automl.logger: 04-04 10:34:57] {2911} INFO - iteration 6, current learner lgbm
[flaml.automl.logger: 04-04 10:34:57] {3097} INFO -  at

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 04-04 10:37:18] {3097} INFO -  at 145.0s,	estimator lrl1's best error=7.7350e-01,	best estimator lgbm's best error=3.0635e-01
[flaml.automl.logger: 04-04 10:37:18] {2911} INFO - iteration 55, current learner lrl1
[flaml.automl.logger: 04-04 10:37:21] {3097} INFO -  at 147.5s,	estimator lrl1's best error=7.7350e-01,	best estimator lgbm's best error=3.0635e-01
[flaml.automl.logger: 04-04 10:37:21] {2911} INFO - iteration 56, current learner xgboost
[flaml.automl.logger: 04-04 10:37:22] {3097} INFO -  at 148.6s,	estimator xgboost's best error=4.3422e-01,	best estimator lgbm's best error=3.0635e-01
[flaml.automl.logger: 04-04 10:37:22] {2911} INFO - iteration 57, current learner lgbm
[flaml.automl.logger: 04-04 10:37:31] {3097} INFO -  at 157.4s,	estimator lgbm's best error=3.0570e-01,	best estimator lgbm's best error=3.0570e-01
[flaml.automl.logger: 04-04 10:37:31] {2911} INFO - iteration 58, current learner lrl1
[flaml.automl.logger: 04-04 10:37:33] {3097} INFO -  a

In [23]:
import os
import pandas as pd
import numpy as np

WINDOW_SIZE = 480

def load_nested_test_data_fast(root_path):
    print(f'Searching for CSVs in {root_path}...')
    csv_paths = []
    for root, _, files in os.walk(root_path):
        for f in files:
            if f.endswith('.csv'):
                csv_paths.append(os.path.join(root, f))

    if not csv_paths:
        return pd.DataFrame()

    print(f'Processing {len(csv_paths)} files efficiently...')
    all_rows = []

    for path in sorted(csv_paths):
        df = pd.read_csv(path)
        patient_id = os.path.splitext(os.path.basename(path))[0]

        # Create a dictionary for the segment features
        # We calculate stats directly to avoid the overhead of a full groupby on tiny dataframes
        row = {'patient_id': patient_id}
        for col in ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']:
            if col in df.columns:
                vals = df[col].values
                row[f'{col}_mean'] = np.mean(vals)
                row[f'{col}_std'] = np.std(vals)
                row[f'{col}_min'] = np.min(vals)
                row[f'{col}_max'] = np.max(vals)
                row[f'{col}_median'] = np.median(vals)

        all_rows.append(row)

    return pd.DataFrame(all_rows)

if 'automl' in locals():
    test_seg = load_nested_test_data_fast('/content/test_segment')

    if not test_seg.empty:
        # Ensure columns match the training set order
        X_test = test_seg.drop(['patient_id'], axis=1)
        # Match feature order if necessary
        if 'X_train' in locals():
            X_test = X_test[X_train.columns]

        test_preds_flaml = automl.predict(X_test)
        sample_sub = pd.read_csv('sample_submission.csv')
        submission_flaml = pd.DataFrame({
            'id': sample_sub['id'],
            'Sleep_Stage': test_preds_flaml
        })
        submission_flaml.to_csv('submission_flaml.csv', index=False)
        print('FLAML Submission saved as submission_flaml.csv')
        display(submission_flaml.head())
    else:
        print('Error: Could not load test data.')
else:
    print('Error: automl model not found. Please run the FLAML training cell first.')

Searching for CSVs in /content/test_segment...
Processing 7832 files efficiently...
FLAML Submission saved as submission_flaml.csv


,id,Sleep_Stage
0,test001_00000,W
1,test001_00001,N2
2,test001_00002,N2
3,test001_00003,N2
4,test001_00004,N2


Score: 0.44946

Private score: